In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns
from PIL import Image
import cv2
import os
from tqdm import tqdm

def main():
    """Main function to run the entire training pipeline"""
    
    print("="*70)
    print("🚀 BINARY IMAGE CLASSIFIER - REAL vs FAKE")
    print("="*70)
    
    # ============================================================================
    # 1. SETUP AND CONFIGURATION
    # ============================================================================
    print("\n📋 Step 1: Configuration Setup")
    print("-"*70)
    
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✅ Using device: {device}")
    
    # Configuration
    dataset_path = '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'  # Updated to match your structure
    train_path = os.path.join(dataset_path, 'train')
    val_path = os.path.join(dataset_path, 'valid')
    test_path = os.path.join(dataset_path, 'test')
    
    batch_size = 32
    num_epochs = 20
    learning_rate = 1e-3
    patience = 5
    
    # ============================================================================
    # 2. DATA PREPROCESSING
    # ============================================================================
    print("\n📊 Step 2: Data Preprocessing")
    print("-"*70)
    
    # Define transforms with data augmentation for training
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Define transforms for validation (no augmentation)
    val_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("✅ Transforms created (with data augmentation for training)")
    
    # ============================================================================
    # 3. LOAD DATASET
    # ============================================================================
    print("\n📁 Step 3: Loading Dataset")
    print("-"*70)
    
    # Check if dataset exists
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset path '{dataset_path}' not found!")
    
    # Load datasets directly from train/valid folders
    if os.path.exists(train_path) and os.path.exists(val_path):
        print("✅ Using pre-split train/valid folders")
        train_dataset = datasets.ImageFolder(root=train_path, transform=train_transform)
        val_dataset = datasets.ImageFolder(root=val_path, transform=val_transform)
        
        class_names = train_dataset.classes
        print(f"✅ Classes found: {class_names}")
        print(f"✅ Training samples: {len(train_dataset)}")
        print(f"✅ Validation samples: {len(val_dataset)}")
        
        # Get class distribution
        train_labels = train_dataset.targets
        val_labels = val_dataset.targets
        
        print(f"✅ Train class distribution: {dict(zip(class_names, np.bincount(train_labels)))}")
        print(f"✅ Val class distribution: {dict(zip(class_names, np.bincount(val_labels)))}")
    else:
        raise FileNotFoundError(f"Train or valid folders not found in '{dataset_path}'")
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    # ============================================================================
    # 4. MODEL CREATION
    # ============================================================================
    print("\n🏗️  Step 4: Creating Model")
    print("-"*70)
    
    # Enhanced Model Architecture
    class ImprovedResNet(nn.Module):
        def __init__(self, num_classes=2, dropout_rate=0.5):
            super(ImprovedResNet, self).__init__()
            self.resnet = models.resnet18(pretrained=True)
            
            # Freeze early layers
            for param in list(self.resnet.parameters())[:-10]:
                param.requires_grad = False
            
            # Replace final layer
            num_features = self.resnet.fc.in_features
            self.resnet.fc = nn.Sequential(
                nn.Dropout(dropout_rate),
                nn.Linear(num_features, 256),
                nn.ReLU(),
                nn.Dropout(dropout_rate/2),
                nn.Linear(256, num_classes)
            )
            
            # Initialize new layers
            for m in self.resnet.fc.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    nn.init.constant_(m.bias, 0)
        
        def forward(self, x):
            return self.resnet(x)
    
    model = ImprovedResNet(num_classes=2, dropout_rate=0.5)
    model = model.to(device)
    
    print(f"✅ Model created: ResNet18 with custom classifier")
    print(f"✅ Model moved to: {device}")
    
    # ============================================================================
    # 5. LOSS FUNCTION AND OPTIMIZER
    # ============================================================================
    print("\n⚙️  Step 5: Setting up Loss & Optimizer")
    print("-"*70)
    
    # Class weights for imbalanced datasets
    class_counts = np.bincount(train_labels)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    class_weights = torch.FloatTensor(class_weights).to(device)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    # Optimizer with different learning rates
    backbone_params = []
    classifier_params = []
    
    for name, param in model.named_parameters():
        if 'fc' in name:
            classifier_params.append(param)
        else:
            backbone_params.append(param)
    
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': 1e-4},
        {'params': classifier_params, 'lr': learning_rate}
    ], weight_decay=1e-4)
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
    
    print("✅ Loss function: CrossEntropyLoss with class weights")
    print("✅ Optimizer: AdamW with differential learning rates")
    print("✅ Scheduler: ReduceLROnPlateau")
    
    # ============================================================================
    # 6. TRAINING FUNCTIONS
    # ============================================================================
    
    def train_epoch(model, train_loader, criterion, optimizer, device):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        return epoch_loss, epoch_acc
    
    def validate_epoch(model, val_loader, criterion, device):
        model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation", leave=False):
                images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        epoch_loss = running_loss / len(val_loader)
        epoch_acc = 100 * correct / total
        return epoch_loss, epoch_acc
    
    # ============================================================================
    # 7. TRAINING LOOP
    # ============================================================================
    print("\n🏋️  Step 6: Training Model")
    print("-"*70)
    
    train_losses, train_accuracies = [], []
    val_losses, val_accuracies = [], []
    best_val_acc = 0.0
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 50)
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        
        # Validate
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        
        # Scheduler step
        scheduler.step(val_loss)
        
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f"✅ New best validation accuracy: {best_val_acc:.2f}%")
        else:
            patience_counter += 1
            print(f"⏳ Patience: {patience_counter}/{patience}")
            
            if patience_counter >= patience:
                print("🛑 Early stopping triggered!")
                break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n✅ Loaded best model with validation accuracy: {best_val_acc:.2f}%")
    
    # ============================================================================
    # 8. PLOT TRAINING CURVES
    # ============================================================================
    print("\n📈 Step 7: Plotting Training Curves")
    print("-"*70)
    
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss', marker='o')
    plt.plot(val_losses, label='Validation Loss', marker='s')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accuracies, label='Train Accuracy', marker='o')
    plt.plot(val_accuracies, label='Validation Accuracy', marker='s')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150)
    print("✅ Training curves saved as 'training_curves.png'")
    plt.show()
    
    # ============================================================================
    # 9. FINAL EVALUATION
    # ============================================================================
    print("\n🎯 Step 8: Final Evaluation")
    print("-"*70)
    
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Final Evaluation"):
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    print(f"\n✅ Final Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig('confusion_matrix.png', dpi=150)
    print("✅ Confusion matrix saved as 'confusion_matrix.png'")
    plt.show()
    
    # Classification Report
    print("\n📊 Classification Report:")
    print(classification_report(all_labels, all_predictions, target_names=class_names))
    
    # ============================================================================
    # 10. PREDICTION FUNCTIONS
    # ============================================================================
    
    def predict_image_simple(model, image_path, transform, device, class_names):
        """Simple prediction function"""
        try:
            model.eval()
            image = Image.open(image_path).convert('RGB')
            input_tensor = transform(image).unsqueeze(0).to(device)
            
            with torch.no_grad():
                outputs = model(input_tensor)
                probabilities = torch.softmax(outputs, dim=1)
                confidence, predicted = torch.max(probabilities, 1)
                
                predicted_class = class_names[predicted.item()]
                confidence_score = confidence.item()
                fake_prob = probabilities[0][0].item()
                real_prob = probabilities[0][1].item()
                
                reliability = "HIGH" if confidence_score >= 0.7 else "MEDIUM" if confidence_score >= 0.5 else "LOW"
            
            return predicted_class, confidence_score, fake_prob, real_prob, reliability
        except Exception as e:
            print(f"Error: {e}")
            return None, None, None, None, None
    
    def display_predictions(model, image_paths, transform, device, class_names, max_images=6):
        """Display predictions on multiple images"""
        if not image_paths:
            print("No images to display")
            return
        
        num_images = min(len(image_paths), max_images)
        rows = (num_images + 2) // 3
        cols = min(3, num_images)
        
        fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
        if num_images == 1:
            axes = [axes]
        else:
            axes = axes.ravel() if num_images > 1 else [axes]
        
        for i, image_path in enumerate(image_paths[:max_images]):
            try:
                result = predict_image_simple(model, image_path, transform, device, class_names)
                
                if result[0] is not None:
                    predicted_class, confidence, fake_prob, real_prob, reliability = result
                    
                    image = Image.open(image_path).convert('RGB')
                    axes[i].imshow(image)
                    axes[i].axis('off')
                    
                    title = f'Prediction: {predicted_class.upper()}\n'
                    title += f'Confidence: {confidence:.3f} ({reliability})\n'
                    title += f'Real: {real_prob:.3f} | Fake: {fake_prob:.3f}'
                    
                    color = 'green' if predicted_class == 'real' else 'red'
                    axes[i].set_title(title, fontsize=10, color=color, weight='bold')
                else:
                    axes[i].text(0.5, 0.5, 'Error', ha='center', va='center')
                    axes[i].axis('off')
            except Exception as e:
                print(f"Error with {image_path}: {e}")
                axes[i].axis('off')
        
        # Hide unused subplots
        for i in range(num_images, len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.savefig('prediction_results.png', dpi=150)
        print("✅ Predictions saved as 'prediction_results.png'")
        plt.show()
    
    # ============================================================================
    # 11. SAMPLE PREDICTIONS
    # ============================================================================
    print("\n🖼️  Step 9: Sample Predictions")
    print("-"*70)
    
    # Find sample images from validation or test set
    sample_images = []
    search_paths = [val_path, test_path] if os.path.exists(test_path) else [val_path]
    
    for search_path in search_paths:
        if os.path.exists(search_path):
            for root, dirs, files in os.walk(search_path):
                for file in files:
                    if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        sample_images.append(os.path.join(root, file))
                        if len(sample_images) >= 6:
                            break
                if len(sample_images) >= 6:
                    break
        if len(sample_images) >= 6:
            break
    
    if sample_images:
        print(f"✅ Found {len(sample_images)} sample images")
        try:
            display_predictions(model, sample_images, val_transform, device, class_names)
        except Exception as e:
            print(f"⚠️  Warning: Could not display predictions - {e}")
            print("Continuing anyway...")
    else:
        print("⚠️  No sample images found for prediction display")
    
    # ============================================================================
    # 12. GRAD-CAM VISUALIZATION (ERROR-SAFE)
    # ============================================================================
    print("\n🔍 Step 10: Grad-CAM Visualization (Optional)")
    print("-"*70)
    
    class GradCAM:
        def __init__(self, model, target_layer):
            self.model = model
            self.target_layer = target_layer
            self.gradients = None
            self.activations = None
            self.target_layer.register_forward_hook(self.save_activation)
            self.target_layer.register_backward_hook(self.save_gradient)
        
        def save_activation(self, module, input, output):
            self.activations = output
        
        def save_gradient(self, module, grad_input, grad_output):
            self.gradients = grad_output[0]
        
        def generate_cam(self, input_image, class_idx=None):
            model_output = self.model(input_image)
            if class_idx is None:
                class_idx = torch.argmax(model_output, dim=1)
            
            self.model.zero_grad()
            class_score = model_output[:, class_idx]
            class_score.backward()
            
            gradients = self.gradients[0].cpu()
            activations = self.activations[0].cpu()
            weights = torch.mean(gradients, dim=[1, 2])
            
            cam = torch.zeros(activations.shape[1:], dtype=torch.float32)
            for i, w in enumerate(weights):
                cam += w * activations[i]
            
            cam = torch.relu(cam)
            if torch.max(cam) > 0:
                cam = cam / torch.max(cam)
            
            return cam.detach().numpy()
    
    def visualize_gradcam(model, image_path, transform, device, class_names):
        """Visualize Grad-CAM - FULLY ERROR-SAFE"""
        try:
            model.eval()
            image = Image.open(image_path).convert('RGB')
            original_image = np.array(image)
            input_tensor = transform(image).unsqueeze(0).to(device)
            
            gradcam = GradCAM(model, model.resnet.layer4[-1])
            cam = gradcam.generate_cam(input_tensor)
            cam_resized = cv2.resize(cam, (original_image.shape[1], original_image.shape[0]))
            
            heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
            heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
            superimposed = heatmap * 0.4 + original_image * 0.6
            superimposed = np.uint8(superimposed)
            
            with torch.no_grad():
                output = model(input_tensor)
                _, predicted = torch.max(output, 1)
                predicted_class = class_names[predicted.item()]
                confidence = torch.softmax(output, dim=1)[0][predicted].item()
            
            plt.figure(figsize=(15, 5))
            
            plt.subplot(1, 3, 1)
            plt.imshow(original_image)
            plt.title('Original Image')
            plt.axis('off')
            
            plt.subplot(1, 3, 2)
            plt.imshow(cam_resized, cmap='jet')
            plt.title('Grad-CAM Heatmap')
            plt.axis('off')
            
            plt.subplot(1, 3, 3)
            plt.imshow(superimposed)
            plt.title(f'Overlay\nPrediction: {predicted_class}\nConfidence: {confidence:.3f}')
            plt.axis('off')
            
            plt.tight_layout()
            plt.savefig('gradcam_visualization.png', dpi=150)
            print("✅ Grad-CAM saved as 'gradcam_visualization.png'")
            plt.show()
            return True
            
        except Exception as e:
            print(f"⚠️  Grad-CAM generation failed: {e}")
            print("⚠️  This is not critical - continuing without Grad-CAM visualization")
            return False
    
    # Try to generate Grad-CAM (non-blocking)
    gradcam_success = False
    if sample_images:
        print("Attempting Grad-CAM visualization for first sample image...")
        print("(If this fails, the script will continue normally)")
        try:
            gradcam_success = visualize_gradcam(model, sample_images[0], val_transform, device, class_names)
        except Exception as e:
            print(f"⚠️  Grad-CAM completely failed: {e}")
            print("⚠️  Skipping Grad-CAM and continuing...")
    else:
        print("⚠️  No sample images available for Grad-CAM")
    
    if not gradcam_success:
        print("ℹ️  Grad-CAM was skipped or failed - this doesn't affect your trained model")
    
    # ============================================================================
    # 13. SAVE MODEL
    # ============================================================================
    print("\n💾 Step 11: Saving Model")
    print("-"*70)
    
    torch.save(model.state_dict(), 'binary_classifier_resnet18.pth')
    print("✅ Model saved as 'binary_classifier_resnet18.pth'")
    
    # ============================================================================
    # 14. SUMMARY AND USAGE INSTRUCTIONS
    # ============================================================================
    print("\n" + "="*70)
    print("🎉 TRAINING COMPLETED SUCCESSFULLY!")
    print("="*70)
    
    print("\n📁 Files Created:")
    print("  • training_curves.png - Training/validation loss and accuracy")
    print("  • confusion_matrix.png - Confusion matrix visualization")
    print("  • prediction_results.png - Sample predictions")
    print("  • gradcam_visualization.png - Grad-CAM heatmap")
    print("  • binary_classifier_resnet18.pth - Trained model weights")
    
    print(f"\n📊 Final Results:")
    print(f"  • Best Validation Accuracy: {best_val_acc:.2f}%")
    print(f"  • Training completed in {len(train_losses)} epochs")
    
    print("\n🔮 How to Use for Predictions:")
    print("  # Load the model:")
    print("  model = ImprovedResNet()")
    print("  model.load_state_dict(torch.load('binary_classifier_resnet18.pth'))")
    print("  model.eval()")
    print()
    print("  # Make prediction:")
    print("  result = predict_image_simple(model, 'path/to/image.jpg', val_transform, device, class_names)")
    print("  predicted_class, confidence, fake_prob, real_prob, reliability = result")
    print("  print(f'Prediction: {predicted_class} (confidence: {confidence:.3f})')")
    
    print("\n" + "="*70)
    
    return model, val_transform, device, class_names, predict_image_simple

# ============================================================================
# RUN THE ENTIRE PIPELINE
# ============================================================================
if __name__ == "__main__":
    try:
        model, val_transform, device, class_names, predict_fn = main()
        print("\n✅ All steps completed successfully!")
        print("✅ Use the returned 'predict_fn' to make predictions on new images")
    except Exception as e:
        print(f"\n❌ Error during execution: {e}")
        import traceback
        traceback.print_exc()

In [5]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import glob
import pandas as pd

# 1. Device Configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# --- Configuration ---
# **CHANGE THIS TO YOUR FOLDER PATH**
TEST_FOLDER_PATH = '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test/fake'
MODEL_PATH = '/kaggle/input/new-model-18-13/pytorch/default/1' 
CLASS_NAMES = ['Fake', 'Real'] # Must match training class order

# =====================================================================
# 2. Custom Dataset for Prediction (No Labels Required)
# =====================================================================
class PredictionDataset(Dataset):
    """A custom dataset for loading images from a single folder for prediction."""
    def __init__(self, root_dir, transform=None):
        # Get all image paths (e.g., .jpg, .png)
        self.image_paths = glob.glob(os.path.join(root_dir, '*.[jJ][pP][gG]'))
        self.image_paths.extend(glob.glob(os.path.join(root_dir, '*.[pP][nN][gG]')))
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        # We only return the image tensor and its filename (no label)
        return image, os.path.basename(img_path)

# =====================================================================
# 3. Model Definition (as provided in the prompt)
# =====================================================================
class ImprovedResNet(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.5):
        super(ImprovedResNet, self).__init__()
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) 
        
        # Freeze early layers
        for param in list(self.resnet.parameters())[:-10]:
            param.requires_grad = False
        
        # Replace final layer
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate/2),
            nn.Linear(256, num_classes)
        )
        
        # Initialize new layers
        for m in self.resnet.fc.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.resnet(x)

# =====================================================================
# 4. Load Model and Data
# =====================================================================
def load_model(model_path):
    model = ImprovedResNet(num_classes=len(CLASS_NAMES), dropout_rate=0.5)
    
    if os.path.exists(model_path):
        try:
            model.load_state_dict(torch.load(model_path, map_location=device))
            print(f"✅ Successfully loaded trained weights from: {model_path}")
        except Exception as e:
            print(f"❌ Error loading model state dict, using random weights: {e}")
    else:
        print(f"⚠️ Model weights file not found at: {model_path}. Using random weights.")
        
    model = model.to(device)
    model.eval() # Crucial for inference
    return model

# Define Transformations
# This should match the transformations used during training!
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# Create Dataset and DataLoader
if not os.path.exists(TEST_FOLDER_PATH) or not os.listdir(TEST_FOLDER_PATH):
    print(f"\n❌ ERROR: Test folder '{TEST_FOLDER_PATH}' not found or is empty.")
    print("Please create the folder and put some images inside.")
    # Create a dummy folder for demonstration if it doesn't exist
    os.makedirs(TEST_FOLDER_PATH, exist_ok=True)
    # exit() # Uncomment this to stop if you need a real test

prediction_dataset = PredictionDataset(TEST_FOLDER_PATH, transform=data_transforms)
prediction_dataloader = DataLoader(prediction_dataset, batch_size=8, shuffle=False, num_workers=0)

model = load_model(MODEL_PATH)

# =====================================================================
# 5. Prediction and Reporting Function
# =====================================================================
def predict_and_report(model, dataloader, class_names):
    """Runs prediction on all images in the DataLoader and prints/saves results."""
    
    print(f"\n--- Running Prediction on {len(dataloader.dataset)} Images ---")
    results = []
    
    # Iterate through the DataLoader (batches of images)
    for images, filenames in dataloader:
        images = images.to(device)
        
        with torch.no_grad():
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1) 
            
            # Get the predicted class index
            confidence, predicted_indices = torch.max(probabilities, 1)
            
        # Process results for the batch
        for i in range(len(filenames)):
            filename = filenames[i]
            predicted_index = predicted_indices[i].item()
            predicted_class = class_names[predicted_index]
            confidence_score = confidence[i].item() * 100
            
            results.append({
                'Filename': filename,
                'Prediction': predicted_class,
                'Confidence (%)': f"{confidence_score:.2f}"
            })

    # Display results using Pandas for clean output
    if results:
        df = pd.DataFrame(results)
        print("\nPrediction Results:")
        print(df.to_markdown(index=False))
    else:
        print("No images found for prediction.")
        
# =====================================================================
# 6. Run the Test
# =====================================================================
predict_and_report(model, prediction_dataloader, CLASS_NAMES)

❌ Error loading model state dict, using random weights: [Errno 21] Is a directory: '/kaggle/input/new-model-18-13/pytorch/default/1'

--- Running Prediction on 10000 Images ---


KeyboardInterrupt: 